In [12]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np

In [13]:
iris = load_iris()
X, y = iris.data, iris.target

In [14]:
X_1 = X[0:50]
y_1 = y[0:50]

X_2 = X[50:100]
y_2 = y[50:100]

X_3 = X[100:150]
y_3 = y[100:150]

In [15]:
X1_train, X1_test, y1_train, y1_test = train_test_split(X_1, y_1, test_size=0.2, random_state=42)
X2_train, X2_test, y2_train, y2_test = train_test_split(X_2, y_2, test_size=0.2, random_state=42)
X3_train, X3_test, y3_train, y3_test = train_test_split(X_3, y_3, test_size=0.2, random_state=42)

In [16]:
# Y = Z^t * W
#
#     | 1 x10 x11 x13 x14 ...x1d |
#     | 1 x20 x21 x23 x24 ...x2d |
# z = | 1 x30 x31 x33 x34 ...x3d |
#     | 1 x40 x41 x43 x44 ...x4d |
#     | .........................|
#     | 1 xn0 xn1 xn3 xn4 ...xnd |
#
# W = |b w0 w1 w2 w3 w4 .......wd|
# note: the training data is balanced
augmented_col = np.concatenate((np.ones((X1_train.shape[0], 1)), np.ones((X2_train.shape[0] + X3_train.shape[0], 1)) * -1), axis = 0)
# form z, as class I is +ve and other is -ve
z1 = np.concatenate((np.concatenate((X1_train, -X2_train), axis = 0), -X3_train), axis=0)
z1 = np.concatenate((augmented_col, z1), axis = 1)
print(z1.shape)

z2 = np.concatenate((np.concatenate((X2_train, -X1_train), axis = 0), -X3_train), axis=0)
z2 = np.concatenate((augmented_col, z2), axis = 1)

z3 = np.concatenate((np.concatenate((X3_train, -X1_train), axis = 0), -X2_train), axis=0)
z3 = np.concatenate((augmented_col, z3), axis = 1)
# yx is the label, it should be any postive smale value
yx = np.ones((z1.shape[0], 1))


(120, 5)


In [17]:
# W = (Z^t * Z)^-1 * Z^t * Y
# Calculate Weight for class I
W1 = np.matmul(z1.T, z1)
W1 = np.linalg.inv(W1)
W1 = np.matmul(W1, z1.T)
print(W1.shape)
W1 = np.matmul(W1, yx)
print(yx.shape)

# Calculate weight for class II
W2 = np.matmul(z2.T, z2)
W2 = np.linalg.inv(W2)
W2 = np.matmul(W2, z2.T)
W2 = np.matmul(W2, yx)

# Calculate weight for class III
W3 = np.matmul(z3.T, z3)
W3 = np.linalg.inv(W3)
W3 = np.matmul(W3, z3.T)
W3 = np.matmul(W3, yx)

(5, 120)
(120, 1)


In [18]:
# Predict class I, should be +ve
# y = x^t * w
y1_predict = np.matmul(np.concatenate((np.ones((X1_test.shape[0], 1)), X1_test), axis = 1), W1)
print(y1_predict)

# other should be -ve
y_predict = np.matmul(np.concatenate((np.ones((X2_test.shape[0], 1)), X2_test), axis = 1), W1)
print(y_predict)

y_predict = np.matmul(np.concatenate((np.ones((X3_test.shape[0], 1)), X3_test), axis = 1),W1)
print(y_predict)

[[0.76558412]
 [0.86150919]
 [0.64191053]
 [0.66769901]
 [0.93717383]
 [1.02562268]
 [0.77745057]
 [0.61958732]
 [1.21420312]
 [1.03365917]]
[[-0.80849762]
 [-0.75011016]
 [-0.68146151]
 [-0.56517597]
 [-0.62094488]
 [-0.33668674]
 [-0.81344811]
 [-0.57114697]
 [-0.56283395]
 [-0.66605154]]
[[-1.25925315]
 [-1.02153781]
 [-1.3754723 ]
 [-1.03432201]
 [-1.17586491]
 [-0.9952846 ]
 [-0.9434529 ]
 [-1.15569265]
 [-1.32305134]
 [-1.29121812]]


In [19]:
# Predict class II, should be +ve
# y = x^t * w
y2_predict = np.matmul(np.concatenate((np.ones((X2_test.shape[0], 1)), X2_test), axis = 1), W2)
print(y2_predict)

# other should be -ve
y_predict = np.matmul(np.concatenate((np.ones((X1_test.shape[0], 1)), X1_test), axis = 1), W2)
print(y_predict)

y_predict = np.matmul(np.concatenate((np.ones((X3_test.shape[0], 1)), X3_test), axis = 1), W2)
print(y_predict)

[[-1.36942634e-04]
 [ 1.79715797e-01]
 [ 3.66533542e-01]
 [-1.03343045e-01]
 [ 2.98873951e-01]
 [-2.36773434e-02]
 [ 1.24238588e-01]
 [-2.04302923e-01]
 [ 4.96980515e-02]
 [ 3.15060656e-01]]
[[-0.31686724]
 [-0.61089541]
 [-0.30516122]
 [-0.37862067]
 [-0.82444074]
 [-0.87845633]
 [-0.74624027]
 [-0.21689262]
 [-1.14718239]
 [-1.05348399]]
[[-0.04999761]
 [-0.5233181 ]
 [ 0.1801524 ]
 [-0.68284784]
 [-0.74375846]
 [-0.96129764]
 [-0.22073134]
 [-0.12605815]
 [-0.26582531]
 [ 0.64963668]]


In [20]:
# Predict class III, should be +ve
# y = x^t * w
y3_predict = np.matmul(np.concatenate((np.ones((X3_test.shape[0], 1)), X3_test), axis = 1), W3)
print(y3_predict)

# other should be -ve
y_predict = np.matmul(np.concatenate((np.ones((X1_test.shape[0], 1)), X1_test), axis = 1), W3)
print(y_predict)

y_predict = np.matmul(np.concatenate((np.ones((X2_test.shape[0], 1)), X2_test), axis = 1), W3)
print(y_predict)

[[ 0.30925077]
 [ 0.54485591]
 [ 0.1953199 ]
 [ 0.71716986]
 [ 0.91962337]
 [ 0.95658224]
 [ 0.16418423]
 [ 0.28175081]
 [ 0.58887666]
 [-0.35841857]]
[[-1.44871688]
 [-1.25061378]
 [-1.33674931]
 [-1.28907834]
 [-1.11273309]
 [-1.14716635]
 [-1.0312103 ]
 [-1.4026947 ]
 [-1.06702073]
 [-0.98017517]]
[[-0.19136544]
 [-0.42960564]
 [-0.68507203]
 [-0.33148098]
 [-0.67792907]
 [-0.63963591]
 [-0.31079048]
 [-0.22455011]
 [-0.4868641 ]
 [-0.64900912]]


In [ ]:
# use traning data for class I
# y = x^t * w
X1_train = np.concatenate((np.ones((X1_train.shape[0], 1)), X1_train), axis = 1)
y1_predict = np.matmul(X1_train, W1)
print(y1_predict)

# use traning data for class II
y2_predict = np.matmul(np.concatenate((np.ones((X2_train.shape[0], 1)), X2_train), axis = 1), W2)
print(y2_predict)

# use traning data for class II
y3_predict = np.matmul(np.concatenate((np.ones((X3_train.shape[0], 1)), X3_train), axis = 1), W3)
print(y3_predict)

[1.  4.8 3.  1.4 0.1]
[1.  4.9 3.1 1.5 0.1]
(40, 5)
[[0.69627246]
 [0.98592675]
 [0.98807049]
 [0.58680463]
 [0.66096676]
 [0.82984975]
 [0.34834803]
 [1.0046037 ]
 [0.75091812]
 [1.37188538]
 [0.71168242]
 [1.18909494]
 [0.65171149]
 [0.6973957 ]
 [0.86936469]
 [0.95146055]
 [0.84600368]
 [0.92026134]
 [1.32629671]
 [1.01572618]
 [0.67637672]
 [0.78173805]
 [1.04337467]
 [0.69412871]
 [0.97276327]
 [0.80640329]
 [0.7954863 ]
 [0.88617442]
 [0.68535546]
 [0.96837304]
 [1.03776566]
 [1.11072358]
 [1.01983268]
 [0.84609922]
 [0.81125375]
 [0.84936621]
 [0.76997435]
 [1.35619168]
 [0.91699436]
 [0.676756  ]]
[[ 0.70514318]
 [-0.03714507]
 [ 0.50619097]
 [ 0.04707129]
 [ 0.35763056]
 [-0.53017337]
 [-0.12692321]
 [-0.10114473]
 [-0.0650378 ]
 [-0.29360469]
 [-0.12175374]
 [-0.2497892 ]
 [-0.06572658]
 [-0.24910042]
 [ 0.41546372]
 [-0.27010859]
 [ 0.07711442]
 [-0.23795121]
 [ 0.15591961]
 [ 0.1012993 ]
 [ 0.16120242]
 [-0.36430902]
 [-0.26687711]
 [-0.43045908]
 [-0.08922268]
 [-0.1919081

#### Conclusion:
It is clear that Class I is linearly separable, which is why we achieved perfect classification.
However, Classes II and III are not linearly separable. We need a method to separate these classes by increasing the dimensionality using a kernel, or by using a neural network with more than one hidden layer.
This is what we will explore in future PRs (ISA).